# Recoding ingredients

In [1]:
!pip3 install pandas
!pip3 install thefuzz

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import pandas as pd
from thefuzz import process
from thefuzz import fuzz

Read the ingredients dictionary into memory.
Write to a csv file.

TODO:
Edit the csv file.
Read back into memory.

In [5]:
pretty_ingredients = (pd.read_json("~/Downloads/recipe-ingredients-dataset/train.json"))["ingredients"].explode('ingredients').drop_duplicates().to_numpy()

pd.DataFrame(pretty_ingredients).to_csv('~/Downloads/pretty_test.csv')

print(pretty_ingredients)

def find_pretty(ing):
    # Remove desciptors like ", washed and cubed"
    ing = str(ing).split(",")[0].split(";")[0]
    matches = []

    # Check for substring matches
    for pretty_ing in pretty_ingredients:
        if fuzz.partial_ratio(pretty_ing, ing) == 100:
            matches.append(pretty_ing)
    
    # If found, find the longest
    # "2 red onions" would match "red onions" and "onions", but "red onions" is more helpful
    if len(matches) > 0:
        matches.sort(key = len, reverse=True)
        return matches[0]
    
    # If no substring matches found, try fuzzy matching with 85% confidence
    match = process.extractOne(ing, pretty_ingredients)
    if match[1] > 85:
        return match[0]
    return None

['romaine lettuce' 'black olives' 'grape tomatoes' ... 'lop chong'
 'tomato garlic pasta sauce' 'crushed cheese crackers']


Read in the recipes to process.

In [6]:
recipes = pd.read_json("~/Downloads/epirecipes/full_format_recipes.json")

print(recipes.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20130 entries, 0 to 20129
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   directions   20111 non-null  object             
 1   fat          15908 non-null  float64            
 2   date         20111 non-null  datetime64[ns, UTC]
 3   categories   20111 non-null  object             
 4   calories     15976 non-null  float64            
 5   desc         13495 non-null  object             
 6   protein      15929 non-null  float64            
 7   rating       20100 non-null  float64            
 8   title        20111 non-null  object             
 9   ingredients  20111 non-null  object             
 10  sodium       15974 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(5), object(5)
memory usage: 1.7+ MB
None


Get just the ingredients from the recipes.
Match with ingredient dictionary.
Write to csv.

In [7]:
# .head() used for testing since find_pretty does about 5 rows per second
ingredients = recipes['ingredients'].explode('ingredients').to_frame().head(10000)

ingredients['pretty'] = (ingredients.map(find_pretty))['ingredients']

ingredients.to_csv('~/Downloads/test.csv')

# Print no match count
print(ingredients.isnull().sum())



ingredients    1
pretty         4
dtype: int64
